# 04. 効果測定
**目的:** 予測モデルを使った動的な人員配置と、実績分布から算出した固定キャパシティ運用を比較し、
入電数予測の導入効果を試算する。

**このNotebookでわかること:**
- 固定キャパシティ運用（過去実績の75パーセンタイルで人員を固定）における応答率
- 予測モデルベースでキャパシティを都度調整した場合の応答率
- 両シナリオの比較による、予測モデル導入の効果


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import PROCESSED_DATA_DIR

In [ ]:
regi_call_df = pd.read_csv(PROCESSED_DATA_DIR / 'regi_call.csv', parse_dates=['cdr_date'])
predictions_df = pd.read_csv(PROCESSED_DATA_DIR / 'test_predictions.csv', parse_dates=['cdr_date'])
predictions_df.head()

## 固定キャパシティ運用の応答率
ヘルプデスクは入電数の75パーセンタイルをカバーできるキャパシティを保持していると仮定する。
ただし単純に75パーセンタイルを取ると税制改正等の突発的なスパイクを含んでしまうため、7日移動平均で平滑化した系列から算出する。

In [ ]:
regi_call_df['call_num_ma7'] = regi_call_df['call_num'].rolling(window=7, min_periods=1).mean()
fixed_threshold = regi_call_df.loc[regi_call_df['holiday_flag'] == False, 'call_num_ma7'].quantile(0.75)

exceed_fixed = predictions_df[predictions_df['call_num'] > fixed_threshold].copy()
exceed_fixed['exceed_count'] = exceed_fixed['call_num'] - fixed_threshold
exceed_fixed['exceed_rate'] = exceed_fixed['exceed_count'] / exceed_fixed['call_num']
exceed_fixed['exceed_rate_o20'] = exceed_fixed['exceed_rate'] >= 0.2

fixed_call_rate = (1 - exceed_fixed['exceed_rate'].mean()) * 100
fixed_o20_cnt = (exceed_fixed['exceed_rate_o20']).sum()
fixed_o20_mean = exceed_fixed.loc[exceed_fixed['exceed_rate_o20'], 'exceed_rate'].mean()

print(f'【固定キャパシティ】平均応答率：{fixed_call_rate:.2f}%')
print(f'【固定キャパシティ】応答率80%を下回った日数：{fixed_o20_cnt}日')
print(f'【固定キャパシティ】その日の平均呼損率：{(fixed_o20_mean * 100):.2f}%')

## 予測モデルベースの応答率
想定キャパシティを日々の予測値に置き換えた場合の応答率を試算する。

In [ ]:
valid = predictions_df[predictions_df['call_num'] > 0].copy()
exceed_pred = valid[valid['call_num'] > valid['y_pred']].copy()
exceed_pred['exceed_count'] = exceed_pred['call_num'] - exceed_pred['y_pred']
exceed_pred['exceed_rate'] = exceed_pred['exceed_count'] / exceed_pred['call_num']
exceed_pred['exceed_rate_o20'] = exceed_pred['exceed_rate'] >= 0.2

pred_call_rate = (1 - exceed_pred['exceed_rate'].mean()) * 100
pred_o20_cnt = (exceed_pred['exceed_rate_o20']).sum()
pred_o20_mean = exceed_pred.loc[exceed_pred['exceed_rate_o20'], 'exceed_rate'].mean()

print(f'【予測モデルベース】平均応答率：{pred_call_rate:.2f}%')
print(f'【予測モデルベース】応答率80%を下回った日数：{pred_o20_cnt}日')
print(f'【予測モデルベース】その日の平均呼損率：{(pred_o20_mean * 100):.2f}%')

## 比較プロット

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(predictions_df['cdr_date'], predictions_df['call_num'], label='実測値', marker='o')
plt.plot(predictions_df['cdr_date'], predictions_df['y_pred'], label='予測値（XGBoost）', marker='o')
plt.axhline(fixed_threshold, color='red', linestyle='--', linewidth=0.5,
            label=f'固定キャパシティ運用時の想定対応キャパ：{fixed_threshold:.1f}')
plt.title('入電数予測と想定対応キャパシティ')
plt.xlabel('日付')
plt.ylabel('入電数')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## まとめ

固定キャパシティ運用（過去実績の75パーセンタイルで人員を固定）に対して、予測モデルに基づき
日々のキャパシティを調整することで、応答率と対応漏れ日数の両面で改善が見込める。
特に連休明けや税制改正・障害発生時などスパイクが起きやすい日を事前に検知できる点が、
固定キャパシティ運用にはない予測モデル導入のメリットである。